**Serverless** means you write code and AWS handles everything else — provisioning servers, patching, scaling, and high availability. You pay only for the compute time your code actually uses.

**AWS Lambda** is the core of serverless compute on AWS. It runs your code in response to events, scales automatically from zero to thousands of concurrent executions, and charges per millisecond of execution time.

## Serverless vs Traditional Compute

| | EC2 | Serverless (Lambda) |
|---|---|---|
| **Provisioning** | You launch and manage instances | AWS manages all infrastructure |
| **Scaling** | Manual or ASG-based | Automatic, instant |
| **Availability** | You design for HA across AZs | Built-in |
| **Cost model** | Pay per hour (running or idle) | Pay per request + execution duration |
| **Idle cost** | Yes — instances run 24/7 | No — zero cost at zero traffic |
| **OS patching** | Your responsibility | AWS responsibility |
| **Max runtime** | Unlimited | 15 minutes per invocation |

> Serverless is not always better — it excels at event-driven, intermittent, or unpredictable workloads. Long-running, stateful, or CPU-intensive jobs often suit EC2 or containers better.

## How Lambda Works

```
Event Source          Lambda Service           Your Code
(S3, API GW,    →    [trigger]    →    Execution Environment
 SQS, etc.)                             ├── Runtime (Python, Node…)
                                         ├── Your function code
                                         ├── Environment variables
                                         └── /tmp (ephemeral storage)
```

When an event triggers your function:
1. Lambda finds or creates an **execution environment** (a sandboxed micro-VM)
2. Initialises your runtime and runs any **init code** (outside the handler)
3. Calls your **handler function** with the event payload
4. Returns the response (or writes to a destination)
5. Keeps the environment warm for a short period to reuse for the next invocation

### Handler signature

```python
def handler(event, context):
    # event  — the trigger payload (dict)
    # context — runtime info: function name, remaining time, request ID
    print(context.function_name)
    print(context.get_remaining_time_in_millis())
    return {'statusCode': 200, 'body': 'Hello'}
```

## Runtimes & Deployment

### Supported runtimes
Python, Node.js, Java, Go, Ruby, .NET (C#/PowerShell), and any language via a **custom runtime** (Amazon Linux 2023-based).

### Deployment options

| Option | Max size | Best for |
|---|---|---|
| **Zip (inline)** | 3 MB (console) | Simple functions, quick edits |
| **Zip via S3** | 250 MB unzipped | Standard deployments |
| **Container image** | 10 GB | Large dependencies, ML models, custom runtimes |

### Lambda Layers
A **layer** is a zip archive of libraries, dependencies, or custom runtimes that you attach to functions. Up to 5 layers per function. Layers are versioned and can be shared across functions or accounts.

```
Function code (your logic)
  + Layer 1 (numpy, pandas)
  + Layer 2 (company shared utilities)
  = Execution environment
```

## Lambda Configuration

| Setting | Range / Options | Notes |
|---|---|---|
| **Memory** | 128 MB – 10,240 MB (in 1 MB steps) | CPU is allocated proportionally to memory |
| **Timeout** | 1 sec – 900 sec (15 min) | Function is terminated if exceeded |
| **Ephemeral storage** (`/tmp`) | 512 MB – 10,240 MB | Shared across warm invocations in same environment |
| **Environment variables** | Up to 4 KB total | Encrypted at rest via KMS |
| **Concurrency** | Up to 1,000 per region (default soft limit) | Can be increased via support ticket |

### CPU allocation
You cannot set CPU directly — it scales linearly with memory. A function at 1,769 MB gets one full vCPU. Functions above that get proportionally more. For CPU-bound tasks, increase memory even if you don't need the RAM.

## Cold Starts vs Warm Starts

```
Cold start:   [download code] → [start runtime] → [run init code] → [handler]
Warm start:                                                          [handler]
```

A **cold start** occurs when Lambda has to create a new execution environment — either the first invocation, after a period of inactivity, or when scaling out to handle more concurrent requests.

Cold start latency varies:
- Python / Node.js: ~100–200 ms
- Java / .NET: ~500 ms – 1 s+ (JVM/CLR startup)
- Container image: potentially longer for large images

### Reducing cold starts

| Technique | How |
|---|---|
| **Provisioned Concurrency** | Pre-warms a fixed number of environments — no cold starts on those |
| **Smaller deployment packages** | Faster code download |
| **Keep init code lean** | Move heavy initialisation outside the handler (but it runs once per environment) |
| **Choose lighter runtimes** | Python/Node cold start faster than Java |

### Execution context reuse
The execution environment stays warm for a period after an invocation. Init code (outside the handler) runs only once per environment — reuse this for expensive setup like DB connections, SDK clients, and loaded ML models.

## Invocation Types & Event Sources

### Synchronous invocation
The caller waits for the function to complete and receives the response.

Sources: **API Gateway**, **ALB**, **Lambda Function URLs**, direct SDK/CLI invocation, Cognito, Lex, Alexa.

On error: caller receives the error response directly. Lambda does **not** retry automatically.

### Asynchronous invocation
Lambda queues the event and returns immediately to the caller. Lambda retries automatically on failure (up to 2 retries with exponential backoff).

Sources: **S3 events**, **SNS**, **EventBridge**, **CloudWatch Logs**, CodeCommit, SES.

On failure after retries: send to a **Dead Letter Queue** (SQS or SNS) or a **Lambda Destination**.

### Event source mapping (poll-based)
Lambda polls the source and invokes your function in batches.

Sources: **SQS**, **Kinesis Data Streams**, **DynamoDB Streams**, **MSK**, **Kafka**.

| Source | Ordering | Scaling |
|---|---|---|
| SQS Standard | Not preserved | One Lambda per batch; scales up to 1,000 concurrent |
| SQS FIFO | Preserved per group | One Lambda per message group |
| Kinesis / DynamoDB Streams | Preserved per shard | One Lambda per shard |

## Concurrency

**Concurrency** = number of function instances running at the same time.

```
Account concurrency limit (default: 1,000 per region)
  ├── Function A: reserved concurrency = 300
  ├── Function B: reserved concurrency = 200
  └── Unreserved pool: 500 (shared by all other functions)
```

### Reserved concurrency
- Guarantees a specific function always has capacity
- **Caps** the function at that limit — useful to protect downstream resources (e.g. limit DB connections)
- Setting reserved concurrency to 0 effectively **disables** the function

### Provisioned concurrency
- Pre-initialises a set number of execution environments
- Those environments are always warm — **no cold start** on requests handled by them
- You pay for provisioned concurrency even when idle
- Ideal for latency-sensitive, customer-facing APIs

### Throttling
When a function exceeds its concurrency limit, Lambda returns a **throttle error** (HTTP 429). For async invocations, Lambda queues and retries. For sync invocations, the error is returned to the caller immediately.

## Lambda Permissions

Lambda uses two IAM concepts:

| | Execution Role | Resource-based Policy |
|---|---|---|
| **What it controls** | What Lambda can do (call S3, DynamoDB, etc.) | Who can invoke this Lambda |
| **Attached to** | The Lambda function (assumed at runtime) | The Lambda function resource |
| **Example** | `AmazonDynamoDBFullAccess` on execution role | Allow API Gateway to invoke the function |

The execution role is the IAM role Lambda assumes when it runs. Always follow least privilege — only grant the specific actions the function needs.

Resource-based policies are added automatically when you add a trigger (API Gateway, S3, etc.) — you can also set them manually for cross-account invocations.

## Lambda Networking

By default, Lambda runs **outside your VPC** in an AWS-managed network. It has internet access and can call public AWS service endpoints.

### VPC-connected Lambda
Configure Lambda to run inside your VPC when it needs to access **private resources** — RDS, ElastiCache, internal services.

```
Lambda (in your VPC subnet)
  ├── Can access: RDS, ElastiCache, internal APIs
  └── Cannot access: internet (by default)
         └── Fix: route through NAT Gateway for internet access
```

### Trade-offs
| | Outside VPC | Inside VPC |
|---|---|---|
| Internet access | Yes | No (unless NAT Gateway) |
| Private resource access | No | Yes |
| Cold start latency | Lower | Slightly higher (ENI attachment) |

> AWS now uses **Hyperplane ENIs** — shared across functions in the same subnet — so VPC cold start overhead is much reduced compared to earlier Lambda versions.

## Error Handling & Destinations

### Lambda Destinations (async invocations)
Route invocation results to another service — for both success and failure:

```
Lambda Function
  ├── On success → SQS / SNS / EventBridge / another Lambda
  └── On failure → SQS / SNS / EventBridge / another Lambda
```

Destinations carry the full event + response payload, making them more useful than DLQs for debugging.

### Dead Letter Queue (DLQ)
For async invocations, after all retries are exhausted, send the failed event to an SQS queue or SNS topic for manual inspection or reprocessing.

### Retry behaviour summary

| Invocation type | Retries on failure |
|---|---|
| Synchronous | 0 — error returned to caller |
| Asynchronous | Up to 2 retries with exponential backoff |
| SQS event source | Retries until message expires or DLQ receives it |
| Kinesis / DynamoDB Streams | Retries until record expires (24h) or bisect-on-error |


## Lambda@Edge & CloudFront Functions

**Lambda@Edge** runs your Lambda functions at CloudFront edge locations — closer to users, without going back to your origin region.

```
User → CloudFront Edge
              ├── Viewer Request  → Lambda@Edge (inspect/modify before cache lookup)
              ├── Origin Request  → Lambda@Edge (before forwarding to origin)
              ├── Origin Response → Lambda@Edge (before caching the response)
              └── Viewer Response → Lambda@Edge (before returning to user)
```

Use cases: A/B testing, authentication at the edge, URL rewrites, personalised responses.

**CloudFront Functions** are lighter-weight (sub-millisecond) JS functions for simple transforms — header manipulation, URL redirects. Cheaper and faster than Lambda@Edge for simple tasks.

| | Lambda@Edge | CloudFront Functions |
|---|---|---|
| Runtime | Node.js, Python | JavaScript only |
| Max duration | 5–30 sec | < 1 ms |
| Network access | Yes | No |
| Use for | Complex logic, auth, API calls | Header tweaks, URL rewrites |

## The Serverless Ecosystem on AWS

Lambda rarely works alone. The full serverless stack:

| Service | Role |
|---|---|
| **API Gateway** | HTTP/WebSocket API front door for Lambda |
| **Lambda Function URLs** | Simple HTTPS endpoint directly on a Lambda (no API Gateway needed) |
| **DynamoDB** | Serverless NoSQL database — scales to zero, pay per request |
| **S3** | Serverless object storage — triggers Lambda on events |
| **SQS / SNS** | Serverless messaging — decouples Lambda producers and consumers |
| **EventBridge** | Serverless event bus — route events from AWS services or custom apps |
| **Step Functions** | Serverless orchestration — coordinate multi-step Lambda workflows |
| **Fargate** | Serverless containers — no EC2 to manage, runs Docker images |
| **Aurora Serverless v2** | Serverless relational DB — scales capacity automatically |
| **Cognito** | Serverless auth — user sign-up, sign-in, JWT tokens |

### Typical serverless API pattern

```
Client
  │
  ▼
API Gateway (HTTP API)
  │
  ▼
Lambda (business logic)
  ├── DynamoDB (data)
  ├── S3 (files)
  └── SQS (async tasks)
```

## Working with Lambda via boto3

In [ ]:
import boto3
import json
import zipfile
import io

lambda_client = boto3.client('lambda', region_name='us-east-1')

In [ ]:
# Package a simple function as a zip in memory
code = b"""
import json

def handler(event, context):
    name = event.get('name', 'World')
    return {
        'statusCode': 200,
        'body': json.dumps({'message': f'Hello, {name}!'})
    }
"""

buf = io.BytesIO()
with zipfile.ZipFile(buf, 'w') as zf:
    zf.writestr('handler.py', code)
buf.seek(0)

response = lambda_client.create_function(
    FunctionName='demo-hello',
    Runtime='python3.12',
    Role='arn:aws:iam::123456789012:role/lambda-execution-role',  # replace
    Handler='handler.handler',
    Code={'ZipFile': buf.read()},
    Timeout=30,
    MemorySize=128,
    Environment={'Variables': {'STAGE': 'dev'}}
)
print(f"Created: {response['FunctionArn']}")

In [ ]:
# Invoke synchronously
response = lambda_client.invoke(
    FunctionName='demo-hello',
    InvocationType='RequestResponse',   # synchronous
    Payload=json.dumps({'name': 'Alice'})
)
result = json.loads(response['Payload'].read())
print(result)

In [ ]:
# Invoke asynchronously — fire and forget
response = lambda_client.invoke(
    FunctionName='demo-hello',
    InvocationType='Event',             # asynchronous
    Payload=json.dumps({'name': 'Bob'})
)
print(f"Status: {response['StatusCode']}")  # 202 = accepted

In [ ]:
# Set reserved concurrency — cap function at 50 concurrent executions
lambda_client.put_function_concurrency(
    FunctionName='demo-hello',
    ReservedConcurrentExecutions=50
)
print("Reserved concurrency set to 50")

In [ ]:
# List all Lambda functions with runtime and memory
paginator = lambda_client.get_paginator('list_functions')
for page in paginator.paginate():
    for fn in page['Functions']:
        print(f"{fn['FunctionName']:<40} {fn['Runtime']:<15} {fn['MemorySize']} MB  "
              f"{fn['Timeout']}s")

In [ ]:
# Add an SQS event source mapping (poll-based trigger)
response = lambda_client.create_event_source_mapping(
    EventSourceArn='arn:aws:sqs:us-east-1:123456789012:demo-queue',  # replace
    FunctionName='demo-hello',
    BatchSize=10,
    MaximumBatchingWindowInSeconds=5,
    FunctionResponseTypes=['ReportBatchItemFailures']  # partial batch success
)
print(f"Event source mapping UUID: {response['UUID']}")

## Summary

| Concept | Key Takeaway |
|---|---|
| Serverless | No server management; pay per use; zero idle cost |
| Lambda handler | `handler(event, context)` — event is the trigger payload |
| Memory | 128 MB – 10 GB; CPU scales proportionally; increase memory for CPU-bound tasks |
| Timeout | Max 15 minutes; not suitable for long-running jobs |
| Cold start | New environment init overhead; worse for Java/.NET; solved by Provisioned Concurrency |
| Sync invocation | Caller waits; no auto-retry; API Gateway, ALB |
| Async invocation | Fire-and-forget; up to 2 retries; S3, SNS, EventBridge |
| Event source mapping | Lambda polls; batches; SQS, Kinesis, DynamoDB Streams |
| Reserved concurrency | Guarantees + caps; set to 0 to disable |
| Provisioned concurrency | Pre-warmed; eliminates cold starts; costs money at idle |
| VPC Lambda | Needed for private resources; needs NAT GW for internet; Hyperplane ENIs reduce cold start |
| Layers | Shared libraries/deps across functions; up to 5 per function |
| Lambda@Edge | Run at CloudFront edge for low-latency transforms and auth |